### Notebook Overview - Feature Engineering

This notebook serves as the bridge between the initial Exploratory Data Analysis (EDA) and the Modeling phase. Following the findings in 01_eda.ipynb, we are now transforming the raw Spotify data into a format optimized for Machine Learning algorithms.

#### Objectives of this Phase:

Data Retrieval: Load the serialized data stored in Parquet format to ensure 100% data type integrity from the EDA stage.

Feature Construction: Create an ad_music_ratio to capture the relationship between weekly ad exposure and daily listening habits.

Categorical feature Encoding: Transform nominal variables (gender, country, subscription_type, device_type) into numerical format using One-Hot Encoding.

Feature Scaling: Standardize numerical features using StandardScaler to ensure that variables with larger ranges (like listening_time) do not disproportionately influence models like KNN or Logistic Regression.

Data Save: Save the final processed features (X) and target (y) for use in the Baseline Modeling notebook.

### Initial Setup & Data Loading

Load data from the paraquet file


In [25]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# Load the cleaned data from the EDA stage
df = pd.read_parquet('/Users/Sucharitha/Desktop/Course/Modules/Module24/SpotifyChurnAnalysis/data/spotify_eda_cleaned.parquet')

# Confirm data types and structure
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                8000 non-null   int64  
 1   gender                 8000 non-null   object 
 2   age                    8000 non-null   int64  
 3   country                8000 non-null   object 
 4   subscription_type      8000 non-null   object 
 5   listening_time         8000 non-null   int64  
 6   songs_played_per_day   8000 non-null   int64  
 7   skip_rate              8000 non-null   float64
 8   device_type            8000 non-null   object 
 9   ads_listened_per_week  8000 non-null   int64  
 10  offline_listening      8000 non-null   int64  
 11  is_churned             8000 non-null   int64  
dtypes: float64(1), int64(7), object(4)
memory usage: 750.1+ KB


### Feature Selection

Removing user_id from the modeling dataset. As a unique identifier, it contains no predictive power so we will remove the feature.

In [26]:
# 1. Drop the user_id column as we cannot use it for prediction
df_model = df.drop(columns=['user_id'])

# 2. Domain Feature Construction: Ad-to-Music Ratio
# We multiply daily listening_time by 7 to get a weekly estimate to match ads_listened_per_week
# Adding 1 to denominator to prevent division by zero
df_model['ad_music_ratio'] = df_model['ads_listened_per_week'] / ((df_model['listening_time'] * 7) + 1)

# 3. Categorical Encoding (One-Hot Encoding)
# This handles gender, country, subscription_type, and device_type
df_encoded = pd.get_dummies(
    df_model,
    columns=['gender', 'country', 'subscription_type', 'device_type'],
    drop_first=True,
    dtype=int)  # This forces 0/1 instead of True/False

# 4. Final Verification
print(f"Original feature count: {df.shape[1]}")
print(f"New feature count after encoding: {df_encoded.shape[1]}")
df_encoded.head()

Original feature count: 12
New feature count after encoding: 22


,age,listening_time,songs_played_per_day,skip_rate,ads_listened_per_week,offline_listening,is_churned,ad_music_ratio,gender_Male,gender_Other,...,country_FR,country_IN,country_PK,country_UK,country_US,subscription_type_Free,subscription_type_Premium,subscription_type_Student,device_type_Mobile,device_type_Web
0,54,26,23,0.20,31,0,1,0.169399,0,0,...,0,0,0,0,0,1,0,0,0,0
1,33,141,62,0.34,0,1,0,0.000000,0,1,...,0,0,0,0,0,0,0,0,0,1
2,38,199,38,0.04,0,1,1,0.000000,1,0,...,0,0,0,0,0,0,1,0,1,0
3,22,36,2,0.31,0,1,0,0.000000,0,0,...,0,0,0,0,0,0,0,1,1,0
4,29,250,57,0.36,0,1,1,0.000000,0,1,...,0,0,0,0,1,0,0,0,1,0


### Create Polynomial Features

In [27]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# 1. Define Features (X) and Target (y)
# We drop 'is_churned' because we only want to transform the input features
X = df_encoded.drop(columns=['is_churned'])
y = df_encoded['is_churned']

# 2. Initialize PolynomialFeatures
# degree=2 creates squares and interactions (e.g., age * skip_rate)
# adding interaction_only=True to ignore squared features such as age^2
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)

# 3. Fit and Transform the features
X_poly = poly.fit_transform(X)

# 4. Get the new feature names
poly_feature_names = poly.get_feature_names_out(X.columns)

print(type(y))

<class 'pandas.core.series.Series'>


### Scale the features

In [28]:
# Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_poly)

print(f"Original feature count: {X.shape[1]}")
print(f"Final feature count after expansion: {X_scaled.shape[1]}")

Original feature count: 21
Final feature count after expansion: 231


### Save the final features

In [29]:
# Create the DataFrame using the scaled data and the polynomial feature names
X_final_df = pd.DataFrame(X_scaled, columns=poly_feature_names)

# Save it as Parquet to preserve the names and data types
X_final_df.to_parquet('/Users/Sucharitha/Desktop/Course/Modules/Module24/SpotifyChurnAnalysis/data/X_final_processed.parquet', index=False)

# Convert Series to DataFrame to make it Parquet-compatible
y.to_frame().to_parquet('/Users/Sucharitha/Desktop/Course/Modules/Module24/SpotifyChurnAnalysis/data/y_final_target.parquet', index=False)

print("Target variable saved as Parquet.")

Target variable saved as Parquet.
